In [ ]:
import pandas as pd
import seaborn as sns
from scipy import stats
import numpy as np
import matplotlib.pyplot as plt

These steps were followed:
1. State the Null and the Alternative Hypotheses
2. Define your alpha.
3. Collect data that is random and independent
4. Calculate the test result
5. Interpret the test result

Hypothesis testing: Chi-Square Test within the Eniac case study
In this notebook, we perform a chi-square test with the data from the Eniac case study, applying a post-hoc correction to perform pairwise tests and find the true winner.

# 1.&nbsp; State the Null Hypothesis and the Alternative Hypothesis.

Null Hypothesis H0: All versions have the same CTR.

Alternative Hypothesis HA: There is a difference in the CTR for the different versions.

# 2.&nbsp; Select an appropriate significance level alpha ($\alpha$).

It was decided that a relatively high alpha was acceptable in this case

In [ ]:
alpha = 0.05

# 3.&nbsp; Collect data that is random and independent
Let's collect them from the .csv files. 

In [ ]:
pd.set_option('display.max_colwidth', None)

In [ ]:
#sample_a

Clicks are stored in row Shop Now/SeeDeals (column Name), extract the number of clicks from column No. Clicks

In [ ]:
a_clicks = sample_a.loc[sample_a['Name'] == 'SHOP NOW', 'No. clicks'].values[0]

In [ ]:
# using a for loop to get clicks of 4 samples at once
clicks=[]
versions=['SHOP NOW',' SEE DEALS']
for df in [sample_a, sample_b, sample_c, sample_d]:
    clicks_1 = df.loc[df['Name'] .isin(versions), 'No. clicks'].iloc[0]
    clicks.append(clicks_1)
clicks

In [ ]:
#Number of visits
sample_a.loc[1, 'Snapshot information']

In [ ]:
# automatically using regex, manually if completely fine, too!
visits = []
for df in (sample_a, sample_b, sample_c, sample_d):
  # Extracts the numeric count from a string containing one or more digits followed by the word "visits."
    val = df.loc[df['Name'] == 'mySidebar', 'Snapshot information'].str.extract(r'(\d+)\s+visits')[0]
    visits.append(int(val.item()))

visits

In [ ]:
#set up dataFrame/contingency table
observed_results = pd.DataFrame(
    [clicks, visits], columns=["A", "B", "C","D"], index=["Clicks", "Visits"]
)
observed_results.loc['No_clicks'] = observed_results.loc['Visits'] - observed_results.loc['Clicks']
observed_results

In [ ]:
data = observed_results.loc[['Clicks', 'No_clicks']]
data

4. # Calculate the test result

In [ ]:
chisq, pvalue, df, expected = stats.chi2_contingency(data)

In [ ]:
chisq

In [ ]:
pvalue

## 5.&nbsp; Interpret the test result

The p-value is smaller than alpha.
  We reject the null hypothesis. There is sufficient evidence to suggest
  at least one of the web pages has a significantly different click-through rate
  compared to the others. Identify the better-performing website with a post-hoc test.

# How do we decide who's the winner?

In [ ]:
import itertools

combos = list(itertools.combinations(data, 2))

for combo in combos:
  print(f"{combo[0]} vs {combo[1]}", end=': ')
  res = stats.chi2_contingency(data.loc[:,list(combo)])
  if res.pvalue < alpha / len(combos):
    print(f"Significant. p = {res.pvalue:.3f}")
  else:
    print(f"NOT significant. p = {res.pvalue:.3f}")


Calculate and plot CTRs

In [ ]:
# CTR rate
observed_results.loc['CTR'] = observed_results.loc['Clicks'] / observed_results.loc['Visits']
observed_results.loc['CTR'].sort_values(ascending=False)
observed_results

In [ ]:
observed_results.loc['CTR'].sort_values(ascending=False).plot(kind='bar');

In [ ]:
# 1. Calculate Standard Error and Margin of Error (95% CI)
# Formula: 1.96 * sqrt( (p * (1-p)) / n )
z = 1.96
observed_results_T = observed_results.T  # Transpose so versions are rows
p = observed_results_T['CTR']
n = observed_results_T['Visits']

moe = z * np.sqrt((p * (1 - p)) / n)

# 2. Plotting
plt.figure(figsize=(10, 6))
order = p.sort_values(ascending=False).index

ax = sns.barplot(
    x=order,
    y=p.loc[order],
    hue=observed_results_T.index,
    palette=['skyblue' if webp == 'A' else 'lightgrey' for webp in order]
)

# 3. Add the manual error bars
plt.errorbar(
    x=np.arange(len(order)),
    y=p.loc[order],
    yerr=moe.loc[order],
    fmt='none', c='black', capsize=5
)

plt.title('Version A and C have the highest CTR and their 95% Confidence Interval overlaps')
plt.ylabel('Click-Through Rate')
plt.xlabel('Website Version')
plt.show()